In [1]:
from neo4j import GraphDatabase
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import tqdm

In [2]:
df = pd.read_csv('fpl_m3.csv')

In [3]:
df = df.rename(columns={'season_x': 'season'})

In [4]:
df['transfers_balance'] = df['transfers_in'] - df['transfers_out']

In [5]:
df['home_team'] = df.apply(lambda x: x['team_x'] if x['was_home'] else x['opp_team_name'], axis=1)
df['away_team'] = df.apply(lambda x: x['opp_team_name'] if x['was_home'] else x['team_x'], axis=1)

In [6]:
df['kickoff_time'] = pd.to_datetime(df['kickoff_time']).dt.strftime('%Y-%m-%d %H:%M:%S+00:00')

In [7]:
df = df.drop(columns=['was_home', 'G_A', 'pos_DEF', 'pos_FWD', 'pos_GK', 'pos_MID', 'team_x_global_code', 'opponent_team_global_code', 
                      'upcoming_total_points', 'season_x_code', 'team_strength', 'opponent_strength', 'strength_difference', 'round', 'opponent_team'])

In [8]:
df.to_csv('fpl_graph_final.csv', index=False)

In [9]:
config = {}
with open("config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        config[key] = value

driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

df = pd.read_csv("fpl_graph_final.csv", header=0)

with driver.session() as session:
    batch_size = 1000

    data = []
    for row in df.itertuples():
        data.append({
            'season': row.season,
            'GW': row.GW,
            'fixture': row.fixture,
            'home_team': row.home_team,
            'away_team': row.away_team,
            'team_x': row.team_x,               # ADDED
            'opp_team_name': row.opp_team_name, # ADDED
            'name': row.name,
            'element': row.element,
            'code': row.code,                   # ADDED
            'position': row.position,
            'kickoff': row.kickoff_time,
            'minutes': row.minutes,
            'goals_scored': row.goals_scored,
            'assists': row.assists,
            'total_points': row.total_points,
            'bonus': row.bonus,
            'clean_sheets': row.clean_sheets,
            'goals_conceded': row.goals_conceded,
            'own_goals': row.own_goals,
            'penalties_saved': row.penalties_saved,
            'penalties_missed': row.penalties_missed,
            'yellow_cards': row.yellow_cards,
            'red_cards': row.red_cards,
            'saves': row.saves,
            'bps': row.bps,
            'influence': row.influence,
            'creativity': row.creativity,
            'threat': row.threat,
            'ict_index': row.ict_index,
            'form': row.form
        })

    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]

        session.run(
            """
            UNWIND $batch as row

            // === NODES ===
            MERGE (s:Season {season_name: row.season})
            MERGE (g:Gameweek {season: row.season, GW_number: row.GW})
            MERGE (f:Fixture {season: row.season, fixture_number: row.fixture})
                SET f.kickoff_time = row.kickoff

            // Teams
            MERGE (t_home:Team {name: row.home_team})
            MERGE (t_away:Team {name: row.away_team})
            MERGE (t_player:Team {name: row.team_x})             // ADDED
            MERGE (t_opp:Team {name: row.opp_team_name})         // ADDED

            // Player
            MERGE (p:Player {
                player_name: row.name,
                code: row.code
            })
        

            // Position
            MERGE (pos:Position {name: row.position})


            // === RELATIONSHIPS ===

            // Season → GW
            MERGE (s)-[:HAS_GW]->(g)

            // GW → Fixture
            MERGE (g)-[:HAS_FIXTURE]->(f)

            // Fixture teams
            MERGE (f)-[:HAS_HOME_TEAM]->(t_home)
            MERGE (f)-[:HAS_AWAY_TEAM]->(t_away)

            // Player → Position
            MERGE (p)-[:PLAYS_AS]->(pos)

            // Player → Team (career/season relationship)
            MERGE (p)-[:PLAYS_FOR {season: row.season}]->(t_player)   // ADDED

            // Opponent link (useful for queries)
            MERGE (p)-[:PLAYED_AGAINST]->(t_opp)                       // ADDED

            // Player → Fixture performance stats
            MERGE (p)-[r:PLAYED_IN]->(f)
            SET
                r.minutes = row.minutes,
                r.goals_scored = row.goals_scored,
                r.assists = row.assists,
                r.total_points = row.total_points,
                r.bonus = row.bonus,
                r.clean_sheets = row.clean_sheets,
                r.goals_conceded = row.goals_conceded,
                r.own_goals = row.own_goals,
                r.penalties_saved = row.penalties_saved,
                r.penalties_missed = row.penalties_missed,
                r.yellow_cards = row.yellow_cards,
                r.red_cards = row.red_cards,
                r.saves = row.saves,
                r.bps = row.bps,
                r.influence = row.influence,
                r.creativity = row.creativity,
                r.threat = row.threat,
                r.ict_index = row.ict_index,
                r.form = row.form
            """,
            batch=batch
        )

        print(f"Imported batch {i // batch_size + 1}: rows {i} to {min(i + batch_size, len(data))}")

    print("Data imported successfully.")

driver.close()


Imported batch 1: rows 0 to 1000
Imported batch 2: rows 1000 to 2000
Imported batch 3: rows 2000 to 3000
Imported batch 4: rows 3000 to 4000
Imported batch 5: rows 4000 to 5000
Imported batch 6: rows 5000 to 6000
Imported batch 7: rows 6000 to 7000
Imported batch 8: rows 7000 to 8000
Imported batch 9: rows 8000 to 9000
Imported batch 10: rows 9000 to 10000
Imported batch 11: rows 10000 to 11000
Imported batch 12: rows 11000 to 12000
Imported batch 13: rows 12000 to 13000
Imported batch 14: rows 13000 to 14000
Imported batch 15: rows 14000 to 15000
Imported batch 16: rows 15000 to 16000
Imported batch 17: rows 16000 to 17000
Imported batch 18: rows 17000 to 18000
Imported batch 19: rows 18000 to 19000
Imported batch 20: rows 19000 to 20000
Imported batch 21: rows 20000 to 21000
Imported batch 22: rows 21000 to 22000
Imported batch 23: rows 22000 to 23000
Imported batch 24: rows 23000 to 24000
Imported batch 25: rows 24000 to 25000
Imported batch 26: rows 25000 to 26000
Imported batch 27

In [11]:
config = {}
with open("config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        config[key] = value

driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

with driver.session() as session:
    result = session.run(
        """
        MATCH (p:Player)-[r:PLAYED_IN]->(f:Fixture)
        OPTIONAL MATCH (p)-[:PLAYS_AS]->(pos:Position)
        OPTIONAL MATCH (p)-[:PLAYS_FOR]->(t:Team)
        RETURN
            p.code AS code,
            p.player_name    AS name,
            coalesce(pos.name, '') AS position,
            coalesce(head(collect(DISTINCT t.name)), '') AS team,
            SUM(r.goals_scored)      AS sum_goals,
            SUM(r.assists)           AS sum_assists,
            SUM(r.total_points)      AS sum_points,
            SUM(r.minutes)           AS sum_minutes,
            SUM(r.clean_sheets)      AS sum_clean_sheets,
            SUM(r.goals_conceded)    AS sum_goals_conceded,
            SUM(r.bps)               AS sum_bps,
            AVG(r.influence)         AS avg_influence,
            AVG(r.creativity)        AS avg_creativity,
            AVG(r.threat)            AS avg_threat,
            AVG(r.ict_index)         AS avg_ict_index
        """
    )

    df = pd.DataFrame(result.data())

driver.close()

print("Rows fetched from Neo4j:", len(df))

Rows fetched from Neo4j: 1340


In [12]:
df.head()

,code,name,position,team,sum_goals,sum_assists,sum_points,sum_minutes,sum_clean_sheets,sum_goals_conceded,sum_bps,avg_influence,avg_creativity,avg_threat,avg_ict_index
0,20467,Theo Walcott,MID,Arsenal,126,77,2212,43274,133,735,6538,6.874074,4.975661,10.798942,2.257143
1,37605,Mesut Özil,MID,Arsenal,36,60,837,15006,60,198,3705,13.053097,25.989381,11.407080,5.049558
2,41792,Aaron Ramsey,MID,Arsenal,16,26,360,5980,24,90,1428,11.389333,10.962667,16.186667,3.849333
3,42427,Kieran Gibbs,DEF,Arsenal,0,12,544,17352,56,288,3164,6.716814,5.275221,2.283186,1.430088
4,44346,Olivier Giroud,FWD,Arsenal,115,30,1200,14635,65,160,4220,6.562914,4.623841,12.403974,2.351656


In [13]:
numeric_cols = [
    "sum_goals",
    "sum_assists",
    "sum_points",
    "sum_minutes",
    "sum_clean_sheets",
    "sum_goals_conceded",
    "sum_bps",
    "avg_influence",
    "avg_creativity",
    "avg_threat",
    "avg_ict_index"
]
# Ensure numeric dtypes
df[numeric_cols] = df[numeric_cols].astype(float)

# Optional: standardize features (recommended)
means = df[numeric_cols].mean()
stds = df[numeric_cols].replace(0, np.nan).std().replace(0, np.nan)

df_norm = (df[numeric_cols] - means) / stds

# Fill NaNs (if any) with 0 after normalization
df_norm = df_norm.fillna(0.0)

# Build list-of-floats vectors
df["embedding_numeric"] = df_norm.apply(lambda row: row.values.astype(float).tolist(), axis=1)

print("Example numeric embedding for first player:")
print(df["embedding_numeric"].iloc[0])
print("Length:", len(df["embedding_numeric"].iloc[0]))

Example numeric embedding for first player:
[2.1602666935802923, 1.7600302103096142, 2.546450571864651, 2.224508496940185, 1.7047705477021047, 2.572937317279771, 1.5660559959908202, 0.25571265420621725, 0.2413089612263, 0.9532532172748948, 0.5950020240344032]
Length: 11


In [14]:
# Build a textual description per player for the text model
def build_player_description(row):
    return (
        f"Player: {row['name']}, "
        f"Position: {row['position']}, "
        f"Team: {row['team']}, "
        f"Goals: {row['sum_goals']}, "
        f"Assists: {row['sum_assists']}, "
        f"Total points: {row['sum_points']}, "
        f"Minutes played: {row['sum_minutes']}, "
        f"Clean sheets: {row['sum_clean_sheets']}, "
        f"Influence: {row['avg_influence']:.1f}, "
        f"Creativity: {row['avg_creativity']:.1f}, "
        f"Threat: {row['avg_threat']:.1f}, "
        f"ICT index: {row['avg_ict_index']:.1f}."
    )

In [15]:
df["description"] = df.apply(build_player_description, axis=1)

print("Sample description:")
print(df["description"].iloc[0])

Sample description:
Player: Theo Walcott, Position: MID, Team: Arsenal, Goals: 126.0, Assists: 77.0, Total points: 2212.0, Minutes played: 43274.0, Clean sheets: 133.0, Influence: 6.9, Creativity: 5.0, Threat: 10.8, ICT index: 2.3.


In [16]:
# Load a HuggingFace embedding model (you can change this later to compare models)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model_v2 = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


# Compute embeddings for all players at once
text_embeddings = model.encode(
    df["description"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

text_embeddings_v2 = model_v2.encode(
    df["description"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Attach to dataframe as list-of-floats
df["embedding_text_l6_v2"] = [vec.astype(float).tolist() for vec in text_embeddings]
df["embedding_text_mpnet_v2"] = [vec.astype(float).tolist() for vec in text_embeddings_v2]

print("Example text embedding for first player:")
print(df["embedding_text_l6_v2"].iloc[0][:10], "...")  # first 10 dims
print("Length:", len(df["embedding_text_l6_v2"].iloc[0]))

print("Example mpnet text embedding for first player:")
print(df["embedding_text_mpnet_v2"].iloc[0][:10], "...")  # first 10 dims
print("Length:", len(df["embedding_text_mpnet_v2"].iloc[0]))

c:\Users\seifd\miniconda3\envs\fpl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\seifd\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back t

Example text embedding for first player:
[-0.01259479857981205, 0.001377182430587709, -0.025190342217683792, -0.022800298407673836, 0.07064932584762573, 0.1416996270418167, 0.08784736692905426, 0.06699339300394058, 0.04762191325426102, 0.0016819064039736986] ...
Length: 384
Example mpnet text embedding for first player:
[-0.06834251433610916, -0.020946791395545006, 0.0025531770661473274, 0.054962776601314545, -0.012917084619402885, -0.03988290950655937, -0.02126964181661606, -7.385032222373411e-05, 0.0007107060519047081, 0.01711978018283844] ...
Length: 768


In [17]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

def write_embeddings(tx, code, emb_numeric, emb_text):
    tx.run(
        """
        MATCH (p:Player {code: $code})
        SET p.embedding_numeric = $embedding_numeric,
            p.embedding_text_l6_v2 = $embedding_text_l6_v2
        """,
        code=code,
        embedding_numeric=emb_numeric,
        embedding_text_l6_v2=emb_text
    )

def write_embeddings_v2(tx, code, emb_text_v2):
    tx.run(
        """
        MATCH (p:Player {code: $code})
        SET p.embedding_text_mpnet_v2 = $embedding_text_mpnet_v2
        """,
        code=code,
        embedding_text_mpnet_v2=emb_text_v2
    )

with driver.session() as session:
    # Iterate through players
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        session.execute_write(
            write_embeddings,
            row["code"],
            row["embedding_numeric"],
            row["embedding_text_l6_v2"]
        )
        session.execute_write(
            write_embeddings_v2,
            row["code"],
            row["embedding_text_mpnet_v2"]
        )

print("Embeddings successfully written to Neo4j!")
driver.close()

100%|██████████| 1340/1340 [00:28<00:00, 46.32it/s]

Embeddings successfully written to Neo4j!


In [18]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))
def create_indexes(tx):
    # Numeric embedding index (dimension = 11)
    tx.run("""
        CREATE VECTOR INDEX playerEmbeddingNumericIndex
        IF NOT EXISTS
        FOR (p:Player) ON (p.embedding_numeric)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 11,
                `vector.similarity_function`: "cosine"
            }
        };
    """)

    # Text embedding index (dimension = 384)
    tx.run("""
        CREATE VECTOR INDEX playerEmbeddingTextIndexL6V2
        IF NOT EXISTS
        FOR (p:Player) ON (p.embedding_text_l6_v2)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 384,
                `vector.similarity_function`: "cosine"
            }
        };
    """)

    tx.run("""
        CREATE VECTOR INDEX playerEmbeddingTextIndexMpnetV2
        IF NOT EXISTS
        FOR (p:Player) ON (p.embedding_text_mpnet_v2)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 768,
                `vector.similarity_function`: "cosine"
            }
        };
    """)

with driver.session() as session:
    session.execute_write(create_indexes)

driver.close()

print("Vector indexes created successfully!")

Vector indexes created successfully!


In [19]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))

with driver.session() as session:
    res = session.run("SHOW INDEXES")
    for row in res:
        print(row)

driver.close()


<Record id=2 name='index_1b9dcc97' state='ONLINE' populationPercent=100.0 type='LOOKUP' entityType='RELATIONSHIP' labelsOrTypes=None properties=None indexProvider='token-lookup-1.0' owningConstraint=None lastRead=None readCount=0>
<Record id=1 name='index_460996c0' state='ONLINE' populationPercent=100.0 type='LOOKUP' entityType='NODE' labelsOrTypes=None properties=None indexProvider='token-lookup-1.0' owningConstraint=None lastRead=neo4j.time.DateTime(2025, 12, 11, 20, 46, 21, 830000000, tzinfo=<UTC>) readCount=856713>
<Record id=3 name='playerEmbeddingNumericIndex' state='POPULATING' populationPercent=0.0 type='VECTOR' entityType='NODE' labelsOrTypes=['Player'] properties=['embedding_numeric'] indexProvider='vector-3.0' owningConstraint=None lastRead=None readCount=None>
<Record id=4 name='playerEmbeddingTextIndexL6V2' state='POPULATING' populationPercent=0.0 type='VECTOR' entityType='NODE' labelsOrTypes=['Player'] properties=['embedding_text_l6_v2'] indexProvider='vector-3.0' owningC

In [20]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))
player = "Mohamed Salah"

query = """
MATCH (p:Player {player_name: $name})
WITH p.embedding_text_l6_v2 AS query_vec
CALL db.index.vector.queryNodes('playerEmbeddingTextIndexL6V2', 5, query_vec)
YIELD node, score
RETURN node.player_name AS similar_player, score
ORDER BY score DESC
"""

with driver.session() as session:
    results = session.run(query, name=player)
    for r in results:
        print(r["similar_player"], r["score"])

driver.close()

Mohamed Salah 1.0
Alireza Jahanbakhsh 0.931881308555603
William Saliba 0.9296044707298279
Ibrahima Konaté 0.9193769693374634
Ishé Samuels-Smith 0.9182944893836975


In [21]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))
player = "Erling Haaland"

query = """
MATCH (p:Player {player_name: $name})
WITH p.embedding_numeric AS query_vec
CALL db.index.vector.queryNodes('playerEmbeddingNumericIndex', 5, query_vec)
YIELD node, score
RETURN node.player_name AS similar_player, score
ORDER BY score DESC
"""

with driver.session() as session:
    results = session.run(query, name=player)
    for r in results:
        print(r["similar_player"], r["score"])

driver.close()

Erling Haaland 1.0
Odsonne Edouard 0.9837496280670166
Miguel Almirón 0.9829938411712646
Miguel Almirón Rejala 0.9829938411712646
Ivan Toney 0.9829584360122681


In [22]:
def get_similar_players(player_name, mode="text", top_k=5):
    if mode == "text":
        embedding_prop = "embedding_text_l6_v2"
        index_name = "playerEmbeddingTextIndexL6V2"
    elif mode == "numeric":
        embedding_prop = "embedding_numeric"
        index_name = "playerEmbeddingNumericIndex"
    elif mode == "text_v2":
        embedding_prop = "embedding_text_mpnet_v2"
        index_name = "playerEmbeddingTextIndexMpnetV2"
    else:
        raise ValueError("mode must be 'text', 'text_v2', or 'numeric'")

    query = f"""
    MATCH (p:Player {{player_name: $name}})
    WITH p.{embedding_prop} AS query_vec
    CALL db.index.vector.queryNodes('{index_name}', $top_k, query_vec)
    YIELD node, score
    RETURN node.player_name AS similar_player, score
    ORDER BY score DESC
    """

    with driver.session() as session:
        results = session.run(query, name=player_name, top_k=top_k)
        return [(r["similar_player"], r["score"]) for r in results]


In [26]:
driver = GraphDatabase.driver(config["URI"], auth=(config["USERNAME"], config["PASSWORD"]))
print(get_similar_players("Mohamed Salah", mode="text"))
print("----"*10)
print(get_similar_players("Mohamed Salah", mode="numeric"))
print("----"*10)
print(get_similar_players("Mohamed Salah", mode="text_v2"))
driver.close()


[('Mohamed Salah', 1.0), ('Alireza Jahanbakhsh', 0.931881308555603), ('William Saliba', 0.9296044707298279), ('Ibrahima Konaté', 0.9193769693374634), ('Ishé Samuels-Smith', 0.9182944893836975)]
----------------------------------------
[('Mohamed Salah', 1.0), ('Sadio Mané', 0.9936059713363647), ('Harvey Barnes', 0.989385724067688), ('Pierre-Emerick Aubameyang', 0.9879378080368042), ('Phil Foden', 0.9866068959236145)]
----------------------------------------
[('Mohamed Salah', 0.9999999403953552), ('Mamadou Sakho', 0.9438453316688538), ('Ibrahima Konaté', 0.9310462474822998), ('Sadio Mané', 0.9303401708602905), ('Roberto Firmino', 0.9240161180496216)]
